# Lab Report - 7
**Experiment No:** 07  
**Experiment Name:** Genetic Algorithm and CSP

In [ ]:
# Lab 7: Genetic Algorithm and CSP
# Run each "# %%" block as a separate cell in Colab/Jupyter, or run the whole file.
# Edit the two lines below so the output shows your details.
MY_NAME = "Laiba Sumaiya Nazim"
MY_ID = "23549009041"

import random
import itertools


def show_id():
    print(f"Name : {MY_NAME},  ID: {MY_ID}")

## Question 1: Exam Timetabling - Problem Setup

In [ ]:
toy_csp_neighbors = {
    'A': ['B'],
    'B': ['A', 'D', 'E', 'F'],
    'C': ['D', 'E'],
    'D': ['B', 'C', 'E', 'F'],
    'E': ['B', 'C', 'D'],
    'F': ['B', 'D'],
}
show_id()
toy_domains = {v: ['S1', 'S2', 'S3'] for v in 'ABCDE'}
toy_domains['F'] = ['S2', 'S3']
variable_order = ['A', 'B', 'C', 'D', 'E', 'F']
value_order = ['S1', 'S2', 'S3']
print('Courses:', variable_order)
print('Allowed slots:', toy_domains)

Name : Laiba Sumaiya Nazim,  ID: 23549009041
Courses: ['A', 'B', 'C', 'D', 'E', 'F']
Allowed slots: {'A': ['S1', 'S2', 'S3'], 'B': ['S1', 'S2', 'S3'], 'C': ['S1', 'S2', 'S3'], 'D': ['S1', 'S2', 'S3'], 'E': ['S1', 'S2', 'S3'], 'F': ['S2', 'S3']}


**Explanation:** The courses, allowed slots and conflicting neighbors are set up as a CSP. Course F cannot take S1.

## 1. Greedy assignment without undoing

In [ ]:
greedy_schedule = {}
show_id()
failed_course = None
for course in variable_order:
    found = False
    for slot in toy_domains[course]:
        if all(greedy_schedule.get(n) != slot for n in toy_csp_neighbors[course]):
            greedy_schedule[course] = slot
            print(course, '=', slot)
            found = True
            break
    if not found:
        failed_course = course
        used = {n: greedy_schedule[n] for n in toy_csp_neighbors[course] if n in greedy_schedule}
        print('Stuck at:', course)
        print('Assigned neighbors:', used)
        break
print('Partial greedy schedule:', greedy_schedule)

Name : Laiba Sumaiya Nazim,  ID: 23549009041
A = S1
B = S2
C = S1
D = S3
Stuck at: E
Assigned neighbors: {'B': 'S2', 'C': 'S1', 'D': 'S3'}
Partial greedy schedule: {'A': 'S1', 'B': 'S2', 'C': 'S1', 'D': 'S3'}


**Explanation:** Greedy takes the first legal slot and never changes earlier choices, so it can get stuck. Here it gets stuck at course E.

## 2. Backtracking search

In [ ]:
exam_attempts = 0
exam_backtracks = 0
def exam_valid(course, slot, assigned):
    return all(assigned.get(n) != slot for n in toy_csp_neighbors[course])
def exam_backtrack(assigned):
    global exam_attempts, exam_backtracks
    if len(assigned) == len(variable_order):
        return assigned.copy()
    course = variable_order[len(assigned)]
    for slot in toy_domains[course]:
        exam_attempts += 1
        if exam_valid(course, slot, assigned):
            assigned[course] = slot
            result = exam_backtrack(assigned)
            if result is not None:
                return result
            del assigned[course]
    exam_backtracks += 1
    return None
exam_solution = exam_backtrack({})
show_id()
print('Final exam schedule:', exam_solution)
print('Assignment attempts:', exam_attempts)
print('Backtracks:', exam_backtracks)

Name : Laiba Sumaiya Nazim,  ID: 23549009041
Final exam schedule: {'A': 'S1', 'B': 'S2', 'C': 'S2', 'D': 'S1', 'E': 'S3', 'F': 'S3'}
Assignment attempts: 17
Backtracks: 2


**Explanation:** Backtracking tries the allowed slots in order and goes back to change an earlier choice when a dead end is reached.

## 3. Check constraints and count all schedules

In [ ]:
show_id()
exam_edges = sorted({tuple(sorted((a, b))) for a in toy_csp_neighbors for b in toy_csp_neighbors[a]})
print('Check every conflicting course pair:')
for a, b in exam_edges:
    ok = exam_solution[a] != exam_solution[b]
    print(f'{a}-{b}: {exam_solution[a]} vs {exam_solution[b]} -> {"Yes" if ok else "No"}')
print('F cannot take S1:', 'Yes' if exam_solution['F'] != 'S1' else 'No')

valid_exam_schedules = []
for values in itertools.product(*[toy_domains[c] for c in variable_order]):
    candidate = dict(zip(variable_order, values))
    if all(candidate[a] != candidate[b] for a, b in exam_edges):
        valid_exam_schedules.append(candidate)
print('Total valid schedules:', len(valid_exam_schedules))
for i, sol in enumerate(valid_exam_schedules[:2], 1):
    print('Valid schedule', i, ':', sol)

Name : Laiba Sumaiya Nazim,  ID: 23549009041
Check every conflicting course pair:
A-B: S1 vs S2 -> Yes
B-D: S2 vs S1 -> Yes
B-E: S2 vs S3 -> Yes
B-F: S2 vs S3 -> Yes
C-D: S2 vs S1 -> Yes
C-E: S2 vs S3 -> Yes
D-E: S1 vs S3 -> Yes
D-F: S1 vs S3 -> Yes
F cannot take S1: Yes
Total valid schedules: 8
Valid schedule 1 : {'A': 'S1', 'B': 'S2', 'C': 'S2', 'D': 'S1', 'E': 'S3', 'F': 'S3'}
Valid schedule 2 : {'A': 'S1', 'B': 'S3', 'C': 'S3', 'D': 'S1', 'E': 'S2', 'F': 'S2'}


**Explanation:** Every conflicting pair and the restriction on F are checked, and all slot combinations are tested to count the valid schedules.

## Question 2: Bangladesh Divisional Map Coloring - Problem Setup

In [ ]:
bd_order = ['Rangpur', 'Rajshahi', 'Mymensingh', 'Khulna', 'Dhaka', 'Barisal', 'Sylhet', 'Chittagong']
bd_domain = ['Red', 'Green', 'Blue']
bd_edges = sorted([
    ('Barisal', 'Chittagong'), ('Barisal', 'Dhaka'), ('Barisal', 'Khulna'),
    ('Chittagong', 'Dhaka'), ('Chittagong', 'Sylhet'),
    ('Dhaka', 'Khulna'), ('Dhaka', 'Mymensingh'), ('Dhaka', 'Rajshahi'), ('Dhaka', 'Sylhet'),
    ('Khulna', 'Rajshahi'),
    ('Mymensingh', 'Rajshahi'), ('Mymensingh', 'Rangpur'), ('Mymensingh', 'Sylhet'),
    ('Rajshahi', 'Rangpur'),
])
bd_neighbors = {d: [] for d in bd_order}
for a, b in bd_edges:
    bd_neighbors[a].append(b)
    bd_neighbors[b].append(a)

**Explanation:** The eight divisions, three colors and 14 borders from the neighbor list are set up as a CSP.

## 1. Genetic algorithm

In [ ]:
random.seed(7)
population_size = 80
generations = 300
mutation_rate = 0.12


def border_conflicts(individual):
    colors = dict(zip(bd_order, individual))
    return sum(colors[a] == colors[b] for a, b in bd_edges)


def random_individual():
    return [random.choice(bd_domain) for _ in bd_order]


def select_parent(population):
    choices = random.sample(population, 3)
    return min(choices, key=border_conflicts)


def crossover(p1, p2):
    cut = random.randint(1, len(bd_order) - 1)
    return p1[:cut] + p2[cut:]


def mutate(child):
    if random.random() < mutation_rate:
        i = random.randrange(len(child))
        child[i] = random.choice([c for c in bd_domain if c != child[i]])
    return child


population = [random_individual() for _ in range(population_size)]
best_ga = min(population, key=border_conflicts).copy()
ga_generations_used = 0
for gen in range(generations + 1):
    current = min(population, key=border_conflicts)
    if border_conflicts(current) < border_conflicts(best_ga):
        best_ga = current.copy()
    ga_generations_used = gen
    if border_conflicts(best_ga) == 0 or gen == generations:
        break
    next_population = [best_ga.copy()]
    while len(next_population) < population_size:
        p1 = select_parent(population)
        p2 = select_parent(population)
        next_population.append(mutate(crossover(p1, p2)))
    population = next_population
show_id()
print('Best GA coloring:', dict(zip(bd_order, best_ga)))
print('Border conflicts:', border_conflicts(best_ga))
print('Generations used:', ga_generations_used)
print('Zero conflicts reached:', border_conflicts(best_ga) == 0)

Name : Laiba Sumaiya Nazim,  ID: 23549009041
Best GA coloring: {'Rangpur': 'Green', 'Rajshahi': 'Blue', 'Mymensingh': 'Red', 'Khulna': 'Red', 'Dhaka': 'Green', 'Barisal': 'Blue', 'Sylhet': 'Blue', 'Chittagong': 'Red'}
Border conflicts: 0
Generations used: 3
Zero conflicts reached: True


**Explanation:** The GA starts with random colorings, picks better parents, combines them with one crossover point and sometimes mutates a color.

## 2. CSP - greedy coloring

In [ ]:
greedy_colors = {}
show_id()
failed_division = None
for division in bd_order:
    allowed = [c for c in bd_domain if all(greedy_colors.get(n) != c for n in bd_neighbors[division])]
    if allowed:
        greedy_colors[division] = allowed[0]
        print(division, '=', allowed[0])
    else:
        failed_division = division
        print('Stuck at:', division)
        print('Already-colored neighbors:',
              {n: greedy_colors[n] for n in bd_neighbors[division] if n in greedy_colors})
        break
print('Greedy result:', greedy_colors)
print('Successful:', failed_division is None)

Name : Laiba Sumaiya Nazim,  ID: 23549009041
Rangpur = Red
Rajshahi = Green
Mymensingh = Blue
Khulna = Red
Stuck at: Dhaka
Already-colored neighbors: {'Khulna': 'Red', 'Mymensingh': 'Blue', 'Rajshahi': 'Green'}
Greedy result: {'Rangpur': 'Red', 'Rajshahi': 'Green', 'Mymensingh': 'Blue', 'Khulna': 'Red'}
Successful: False


**Explanation:** Greedy coloring gives each division the first available color without changing earlier assignments, and it can get stuck.

## 2. CSP - backtracking coloring

In [ ]:
bd_attempts = 0
bd_backtracks = 0


def bd_valid(division, color, assigned):
    return all(assigned.get(n) != color for n in bd_neighbors[division])


def bd_backtrack(assigned):
    global bd_attempts, bd_backtracks
    if len(assigned) == len(bd_order):
        return assigned.copy()
    division = bd_order[len(assigned)]
    for color in bd_domain:
        bd_attempts += 1
        if bd_valid(division, color, assigned):
            assigned[division] = color
            result = bd_backtrack(assigned)
            if result is not None:
                return result
            del assigned[division]
    bd_backtracks += 1
    return None


bd_solution = bd_backtrack({})
show_id()
print('Backtracking coloring:', bd_solution)
print('Assignment attempts:', bd_attempts)
print('Backtracks:', bd_backtracks)

Name : Laiba Sumaiya Nazim,  ID: 23549009041
Backtracking coloring: {'Rangpur': 'Red', 'Rajshahi': 'Green', 'Mymensingh': 'Blue', 'Khulna': 'Blue', 'Dhaka': 'Red', 'Barisal': 'Green', 'Sylhet': 'Green', 'Chittagong': 'Blue'}
Assignment attempts: 20
Backtracks: 1


**Explanation:** Backtracking changes earlier colors when needed. Each color tried and each dead end is counted.

## 2.1 Verifying borders and comparing methods

In [ ]:
show_id()
print('Border verification:')
for a, b in bd_edges:
    ok = bd_solution is not None and bd_solution[a] != bd_solution[b]
    print(f'{a} - {b}: {bd_solution[a] if bd_solution else "-"} / '
          f'{bd_solution[b] if bd_solution else "-"} -> {"Yes" if ok else "No"}')
print('Verified borders:', len(bd_edges))
print('GA generations (Q2):', ga_generations_used)
print('Map CSP assignment attempts:', bd_attempts)
print('Map CSP backtracks:', bd_backtracks)
print('Exam CSP assignment attempts (Q1):', exam_attempts)
print('Note: GA generations and CSP assignment attempts are different types of steps.')

Name : Laiba Sumaiya Nazim,  ID: 23549009041
Border verification:
Barisal - Chittagong: Green / Blue -> Yes
Barisal - Dhaka: Green / Red -> Yes
Barisal - Khulna: Green / Blue -> Yes
Chittagong - Dhaka: Blue / Red -> Yes
Chittagong - Sylhet: Blue / Green -> Yes
Dhaka - Khulna: Red / Blue -> Yes
Dhaka - Mymensingh: Red / Blue -> Yes
Dhaka - Rajshahi: Red / Green -> Yes
Dhaka - Sylhet: Red / Green -> Yes
Khulna - Rajshahi: Blue / Green -> Yes
Mymensingh - Rajshahi: Blue / Green -> Yes
Mymensingh - Rangpur: Blue / Red -> Yes
Mymensingh - Sylhet: Blue / Green -> Yes
Rajshahi - Rangpur: Green / Red -> Yes
Verified borders: 14
GA generations (Q2): 3
Map CSP assignment attempts: 20
Map CSP backtracks: 1
Exam CSP assignment attempts (Q1): 17
Note: GA generations and CSP assignment attempts are different types of steps.


**Explanation:** Every border is verified on the final coloring. GA generations and CSP assignment attempts are different kinds of steps and are not directly comparable.